# Metal checkpoint check
MolScribe fine-tuned on synthetic organometallics (run E1, epoch 6). Needs this branch (`feature/metal-ocsr`).

In [ ]:
import os, sys
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import Draw

sys.path.insert(0, os.path.abspath('..'))  # repo root; the notebook lives in metal_ocsr/
from molscribe import MolScribe
from molscribe.metal import metal_key

CKPT = r'C:\py_code\molscribe_bench\ckpts\metal_E1_ep6.pth'  # server: /mnt/hard1/ivans_data/metal_ocsr/ckpts/E1/metal_E1_ep6.pth
DEVICE = 'cpu'  # 'cuda' on a GPU machine
model = MolScribe(CKPT, device=DEVICE)

In [ ]:
def load_rgb(path):
    """cv2.imread cannot open non-ASCII paths on Windows ("Снимок экрана ..."); decode the bytes instead."""
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    image = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_COLOR)
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)


def draw(smiles, size=450):
    if sys.version_info >= (3, 10):  # metal2d lays out complexes nicely (pip install metal2d)
        try:
            import metal2d
            metal2d.draw(metal2d.depict(Chem.MolFromSmiles(smiles)), '_pred.png', size=(size, size))
            return plt.imread('_pred.png')
        except Exception:
            pass
    mol = Chem.MolFromSmiles(smiles) or Chem.MolFromSmiles(smiles, sanitize=False)
    return Draw.MolToImage(mol, size=(size, size)) if mol is not None else None


def show(path, expected=None):
    image = load_rgb(path)
    smiles = model.predict_image(image)['smiles']
    verdict = '' if expected is None else ('  MATCH' if metal_key(smiles) == metal_key(expected) else '  DIFFERENT')
    fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
    ax[0].imshow(image)
    ax[0].set_title(os.path.basename(path))
    img = draw(smiles)
    if img is not None:
        ax[1].imshow(img)
    ax[1].set_title('prediction' + verdict)
    for a in ax:
        a.axis('off')
    plt.show()
    print(smiles)

In [ ]:
examples = pd.read_csv('examples/examples.csv')
for _, row in examples.iterrows():
    show(os.path.join('examples', row.image), row.expected)

In [ ]:
# your own image
# show(r'C:\path	o\image.png')